In [1]:
"""
OCR date extraction workflow:

1. Tune crop region
2. Compare preprocessing methods
3. Evaluate OCR settings (PSM)
4. Apply regex extraction
5. Normalize OCR errors

Goal: convert noisy flyer text into structured event dates.
"""

from PIL import Image
import matplotlib.pyplot as plt

image_path = "../data/raw_images/dyke_party_01.jpg"

img = Image.open(image_path)

plt.figure(figsize=(8,8))
plt.imshow(img)
plt.axis("off")
plt.show()

<Figure size 800x800 with 1 Axes>

In [14]:
# Cropped Region OCR Test
## Bottom date region
crop = img.crop((0, 900, 1200, 1200))

plt.figure(figsize=(10,4))
plt.imshow(crop, cmap="gray")
plt.axis("off")
plt.show()

<Figure size 1000x400 with 1 Axes>

In [3]:
#Test Text Extraction on Cropped text
import pytesseract

crop_gray = crop.convert("L")
crop_big = crop_gray.resize((crop_gray.width * 2, crop_gray.height * 2))
crop_bw = crop_big.point(lambda p: 255 if p > 160 else 0)

text = pytesseract.image_to_string(crop_bw, config="--psm 6")

print("----- CROPPED OCR OUTPUT -----")
print(repr(text))

----- CROPPED OCR OUTPUT -----
'> ee » Oo & TOTe <\nota ee OE ae abr ag\n'


In [4]:
# More Pre-Processing
import numpy as np
import cv2

img_cv = cv2.imread("../data/raw_images/dyke_party_01.jpg")

# convert to RGB
img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)

# extract red channel
red_channel = img_rgb[:,:,0]

plt.figure(figsize=(10,4))
plt.imshow(red_channel, cmap="gray")
plt.axis("off")

(np.float64(-0.5), np.float64(1078.5), np.float64(1179.5), np.float64(-0.5))

<Figure size 1000x400 with 1 Axes>

In [6]:
#Crop Again
crop = crop_bw.crop((0, 850, 1200, 1050))

plt.figure(figsize=(10,4))
plt.imshow(crop, cmap="gray")
plt.axis("off")

(np.float64(-0.5), np.float64(1199.5), np.float64(199.5), np.float64(-0.5))

<Figure size 1000x400 with 1 Axes>

In [31]:
#All together now

from PIL import Image
import matplotlib.pyplot as plt
import pytesseract
import re

img = Image.open("../data/raw_images/dyke_party_01.jpg")

# Define multiple crop options
crop_boxes = [
    (120, 930, 980, 1080),   # baseline
    (140, 930, 960, 1080),   # trim left + right
    (120, 950, 980, 1070),   # trim top + bottom
    (140, 950, 960, 1070),   # trim all sides
]

# Loop through crops and test OCR
for i, box in enumerate(crop_boxes):
    crop = img.crop(box)

    # Preprocess
    crop_bw = crop.convert("L")
    crop_bw = crop_bw.resize((crop_bw.width * 2, crop_bw.height * 2))

    # OCR
    text = pytesseract.image_to_string(
        crop_bw,
        config="--psm 7 -c tessedit_char_whitelist=0123456789/"
    )

    # 👉 CLEANING STEP GOES HERE
    text = text.replace(" ", "").strip()

    print(f"Test {i}: {box}")
    print("CLEANED OCR:", text)

    # Regex
    match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)
    if match:
        month = int(match.group(1))
        day = int(match.group(2))
        year = match.group(3)

        if 1 <= month <= 12 and 1 <= day <= 31:
            print("VALID DATE FOUND:", f"{month}/{day}/{year}")
        else:
            print("INVALID DATE:", f"{month}/{day}/{year}")
    else:
        print("No clean date found")

    print("-" * 40)

Test 0: (120, 930, 980, 1080)
CLEANED OCR: 02/17/20701
VALID DATE FOUND: 2/17/2070
----------------------------------------
Test 1: (140, 930, 960, 1080)
CLEANED OCR: 62/17/2071
INVALID DATE: 62/17/2071
----------------------------------------
Test 2: (120, 950, 980, 1070)
CLEANED OCR: 6/17/46
VALID DATE FOUND: 6/17/46
----------------------------------------
Test 3: (140, 950, 960, 1070)
CLEANED OCR: 
No clean date found
----------------------------------------


In [32]:
"""
Preprocessing experiment: compare OCR accuracy across image transformations.

Tests multiple preprocessing variants on the same cropped date region:
- grayscale
- grayscale + resize
- grayscale + threshold
- grayscale + threshold + resize

Goal: identify which preprocessing method produces the most accurate OCR output
before applying regex extraction and validation.
"""
from PIL import Image
import pytesseract
import re

img = Image.open("../data/raw_images/dyke_party_01.jpg")

best_box = (120, 930, 980, 1080)
crop = img.crop(best_box)

versions = []

# Version 1: grayscale only
v1 = crop.convert("L")
versions.append(("grayscale", v1))

# Version 2: grayscale + resize
v2 = crop.convert("L")
v2 = v2.resize((v2.width * 2, v2.height * 2))
versions.append(("grayscale_resize", v2))

# Version 3: grayscale + threshold
v3 = crop.convert("L")
v3 = v3.point(lambda p: 255 if p > 160 else 0)
versions.append(("grayscale_threshold", v3))

# Version 4: grayscale + threshold + resize
v4 = crop.convert("L")
v4 = v4.point(lambda p: 255 if p > 160 else 0)
v4 = v4.resize((v4.width * 2, v4.height * 2))
versions.append(("grayscale_threshold_resize", v4))

for name, version in versions:
    text = pytesseract.image_to_string(
        version,
        config="--psm 7 -c tessedit_char_whitelist=0123456789/"
    )

    text = text.replace(" ", "").replace("\n", "").strip()

    print("VERSION:", name)
    print("CLEANED OCR:", text)

    match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)

    if match:
        month = int(match.group(1))
        day = int(match.group(2))
        year = match.group(3)

        if 1 <= month <= 12 and 1 <= day <= 31:
            print("VALID DATE FOUND:", f"{month}/{day}/{year}")
        else:
            print("INVALID DATE:", f"{month}/{day}/{year}")
    else:
        print("No clean date found")

    print("-" * 40)

VERSION: grayscale
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
VERSION: grayscale_resize
CLEANED OCR: 02/17/20701
VALID DATE FOUND: 2/17/2070
----------------------------------------
VERSION: grayscale_threshold
CLEANED OCR: 062/17/20701
INVALID DATE: 62/17/2070
----------------------------------------
VERSION: grayscale_threshold_resize
CLEANED OCR: 
No clean date found
----------------------------------------


In [33]:
# Compare Tesseract PSM modes to see if layout assumptions affect OCR accuracy for date extraction
from PIL import Image
import pytesseract
import re

img = Image.open("../data/raw_images/dyke_party_01.jpg")

best_box = (120, 930, 980, 1080)
crop = img.crop(best_box)

crop_bw = crop.convert("L")

psm_modes = [6, 7, 8, 13]

for psm in psm_modes:
    config = f"--psm {psm} -c tessedit_char_whitelist=0123456789/"
    
    text = pytesseract.image_to_string(crop_bw, config=config)
    text = text.replace(" ", "").replace("\n", "").strip()

    print("PSM:", psm)
    print("CLEANED OCR:", text)

    match = re.search(r"(\d{1,2})/(\d{1,2})/(\d{2,4})", text)

    if match:
        month = int(match.group(1))
        day = int(match.group(2))
        year = match.group(3)

        if 1 <= month <= 12 and 1 <= day <= 31:
            print("VALID DATE FOUND:", f"{month}/{day}/{year}")
        else:
            print("INVALID DATE:", f"{month}/{day}/{year}")
    else:
        print("No clean date found")

    print("-" * 40)

PSM: 6
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
PSM: 7
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
PSM: 8
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------
PSM: 13
CLEANED OCR: 02/17/2077
VALID DATE FOUND: 2/17/2077
----------------------------------------


In [ ]:
## Experiment Summary

Best crop:
(120, 930, 980, 1080) — isolates date region with minimal noise

Best preprocessing:
Grayscale only — resizing and thresholding degraded OCR accuracy

PSM:
No significant difference across modes (6, 7, 8, 13)

Common OCR errors:
Year misread (e.g., 2077 instead of 2026)

Conclusion:
Use grayscale + PSM 7 + whitelist, then correct year via post-processing